In [13]:
# Load env variables
import os
from dotenv import load_dotenv
load_dotenv()
os.environ.pop('ANTHROPIC_AUTH_TOKEN', None)  # workaround: unset empty env var that breaks SDK

# Create an API client
from anthropic import Anthropic
client = Anthropic()
#model = "claude-sonnet-4-6"
#model = "claude-opus-4-6"
model = "claude-haiku-4-5"

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

# Make another request



In [14]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for prompt evaluation. The dataset will be used to evaluate prompt that generate Python, JSON, or regex. Specifically for Python learning tasks, generate an array of JSON, each representing tasks that require Python, JSON, or regex to complete.

Example output:
```json
[
    {
        "tast": "Description of tast",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single pattern function or single JSON object, or regex.
* Focus on tasks that do not require writing much code.
Please generate five objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [15]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [16]:
messages = []
prompt = """
 Generate five different Node.js inverview questions for Senior level. Each should be concise.
"""
add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all questions about Node.js in a single block without any commands:\n```bash")
text = chat(messages, stop_sequences=["```"])
text.strip()


''

In [17]:
messages = []
prompt = """
 Generate three different sample AWS CLI commands. Each should be very short.
"""
add_user_message(messages, prompt)
add_assistant_message(messages, "```")

text = chat(messages, stop_sequences=["```\n"])
text.strip()


'bash\n# 1. List all S3 buckets\naws s3 ls\n\n# 2. Describe EC2 instances\naws ec2 describe-instances\n\n# 3. Get current AWS account ID\naws sts get-caller-identity'

In [18]:
messages = []
add_user_message(messages, " Generate a very short event bridge rule as JSON")
add_assistant_message(messages, "```json")
add_user_message(messages, "not works")
text = chat(messages, stop_sequences=["```"])
text

IndexError: list index out of range

In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="????????")

In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True,
)
for event in stream:
    print(event)

In [ ]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")
# Get the follow-up response with full context
final_answer = chat(messages)
add_assistant_message(messages, final_answer)

messages

In [ ]:
messages = []

#Use a 'while True' loop to run the chatbot forever
while True:
    # Get user input
    user_input = input("> ")
    print("> ", user_input)

    # Add user input to the list of messages
    add_user_message(messages, user_input)
    # Call Claude with the 'chat' function
    answer = chat(messages)
    # Add generated text to the list of messages
    add_assistant_message(messages, answer)
    # Print the generated text
    print("---")
    print(answer)
    print("---")


In [ ]:
messages = []
angryAnswer = """
Answer like You're an angry tired toxic teacher, and don't have enough passion for long conversations. You're motivating people through aggression
"""
politeAnswer = """
You're a patient tutor, that motivating student to find solution by itself.
You help step by step.
"""
add_user_message(messages, "Generate a one sentence small business idea")
answer = chat(messages, system=angryAnswer, temperature=1.0)

print(answer)